In [89]:
from pathlib import Path
import json
import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

In [91]:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "host.docker.internal",
    "port": 27017,
    "database": "shop_db"
}

RAW_DIR = Path("../raw/cleaned")
COLLECTION_NAME = "online_shoppers"

In [93]:
class MongoDatabase:
    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        self.host = db_config["host"]
        self.port = db_config["port"]
        self.user = db_config["user"]
        self.password = db_config["password"]
        self.database_name = db_config["database"]

        self.client = None
        self.db = None

    def connect(self):
        if self.client is None:

            uri = (
                f"mongodb://{self.user}:{self.password}"
                f"@{self.host}:{self.port}/"
                f"?authSource=admin"
            )

            try:
                self.client = MongoClient(
                    uri,
                    serverSelectionTimeoutMS=3000
                )

                self.db = self.client[self.database_name]

                self.client.admin.command("ping")

                print("Connected to MongoDB successfully.")

            except OperationFailure:
                raise Exception(
                    "Authentication failed.\n"
                    "Check the credentials in prj/docker-compose.yml\n"
                    "and test manually with:\n"
                    "docker exec -it lab-mongo mongosh "
                    "-u labuser -p labpass "
                    "--authenticationDatabase admin"
                )

            except ServerSelectionTimeoutError:
                raise Exception(
                    "Could not connect to MongoDB server."
                )

        return self.db

    def is_connected(self):
        return self.client is not None

    def get_collection(self, collection_name):
        return self.db[collection_name]

    def close(self):
        if self.client is not None:
            self.client.close()
            self.client = None
            self.db = None

            print("MongoDB connection closed.")

In [95]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        self.database = database

        self.collection = self.database.get_collection(collection_name)

    def select(self, filter_query=None, projection=None, limit=10):
        if filter_query is None:
            filter_query = {}

        cursor = self.collection.find(
            filter_query,
            projection
        ).limit(limit)

        return list(cursor)

    def insert_one(self, document):
        result = self.collection.insert_one(document)

        return result.inserted_id

    def insert_many(self, documents):
        result = self.collection.insert_many(documents)

        return result.inserted_ids

    def update_one(self, filter_query, update_query):
        result = self.collection.update_one(
                filter_query,
                update_query
        )

        return result.modified_count

    def delete_one(self, filter_query):
        result = self.collection.delete_one(filter_query)

        return result.deleted_count

    def count_documents(self, filter_query=None):
        if filter_query is None:
            filter_query = {}

        return self.collection.count_documents(filter_query)

In [97]:
db = MongoDatabase(MONGO_CONFIG)
db.connect()
executor = MongoExecutor(db, COLLECTION_NAME)
db.client.admin.command("ping")

Connected to MongoDB successfully.


{'ok': 1.0}

In [99]:
online_shoppers_path = RAW_DIR / "online-shoppers-cleaned.csv"
df_shoppers = pd.read_csv(online_shoppers_path, sep=",")
display(df_shoppers.head())

,administrative,administrative-duration,informational,informational duration,productrelated,productrelated duration,bouncerates,exitrates,pagevalues,specialday,month,operatingsystems,browser,region,traffictype,visitortype,weekend,revenue,source_file
0,0.0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,feb,1,1,1,1,returning_visitor,False,False,online-shoppers-dirty.csv
1,0.0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,feb,2,2,1,2,returning_visitor,False,False,online-shoppers-dirty.csv
2,0.0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,feb,4,1,9,3,returning_visitor,False,False,online-shoppers-dirty.csv
3,0.0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,feb,3,2,2,4,returning_visitor,False,False,online-shoppers-dirty.csv
4,0.0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,feb,3,3,1,4,returning_visitor,True,False,online-shoppers-dirty.csv


In [101]:
df_shoppers.dtypes

administrative             float64
administrative-duration    float64
informational                int64
informational duration     float64
productrelated               int64
productrelated duration    float64
bouncerates                float64
exitrates                  float64
pagevalues                 float64
specialday                 float64
month                       object
operatingsystems             int64
browser                      int64
region                       int64
traffictype                  int64
visitortype                 object
weekend                       bool
revenue                       bool
source_file                 object
dtype: object

In [103]:
def map_external_record(record, dataset_name, row_number):
    """Transform one CSV row into one MongoDB document."""

    external_id = f"SESSION-{dataset_name.upper()}-{row_number:05d}"
    
    document = {
        "external_id": external_id,
        "source_dataset": dataset_name,
        "source_file": f"{dataset_name}.csv",

        # numerical features
        "administrative": float(record["administrative"]),
        "administrative_duration": float(record["administrative-duration"]),
        "informational": int(record["informational"]),
        "informational_duration": float(record["informational duration"]),
        "product_related": int(record["productrelated"]),
        "product_related_duration": float(record["productrelated duration"]),
        "bounce_rates": float(record["bouncerates"]),
        "exit_rates": float(record["exitrates"]),
        "page_values": float(record["pagevalues"]),
        "special_day": float(record["specialday"]),
        "operating_systems": int(record["operatingsystems"]),
        "browser": int(record["browser"]),
        "region": int(record["region"]),
        "traffic_type": int(record["traffictype"]),

        # categorical
        "month": record["month"],
        "visitor_type": record["visitortype"],

        # boolean
        "weekend": bool(record["weekend"]),
        "revenue": bool(record["revenue"])
    }

    return document



In [105]:
def ingest_data(executor, csv_path, limit=None):
    df = pd.read_csv(csv_path, sep=",")
    if limit is not None:
        df = df.head(limit)

    dataset_name = csv_path.stem

    documents = [
        map_external_record(row, dataset_name, i)
        for i, row in df.iterrows()
    ]
    inserted_ids = executor.insert_many(documents)

    return len(inserted_ids)

In [125]:
inserted_small = ingest_data(executor, online_shoppers_path, limit=10)
print("Inserted (small batch):", inserted_small)
total_docs = executor.count_documents()
print("Total documents in collection:", total_docs)
sample = executor.select(limit=1)
print(sample)
for doc in executor.select():
    executor.delete_one({"_id": doc["_id"]})

Inserted (small batch): 10
Total documents in collection: 10
[{'_id': ObjectId('6a14dffc94309134984a6d02'), 'external_id': 'SESSION-ONLINE-SHOPPERS-CLEANED-00000', 'source_dataset': 'online-shoppers-cleaned', 'source_file': 'online-shoppers-cleaned.csv', 'administrative': 0.0, 'administrative_duration': 0.0, 'informational': 0, 'informational_duration': 0.0, 'product_related': 1, 'product_related_duration': 0.0, 'bounce_rates': 0.2, 'exit_rates': 0.2, 'page_values': 0.0, 'special_day': 0.0, 'operating_systems': 1, 'browser': 1, 'region': 1, 'traffic_type': 1, 'month': 'feb', 'visitor_type': 'returning_visitor', 'weekend': False, 'revenue': False}]


In [127]:
ingest_data(executor, online_shoppers_path)

12206

In [197]:
#teste
print("Initial count:", executor.count_documents())
test_doc = {
    "test_flag": True,
    "value": 123
}

inserted_id = executor.insert_one(test_doc)
print("Inserted ID:", inserted_id)
print(executor.select({"test_flag": True}))
executor.update_one(
    {"test_flag": True},
    {"$set": {"value": 999}}
)

print("After update:", executor.select({"test_flag": True}))

test_docs = [
    {"test_flag": True, "batch": 1},
    {"test_flag": True, "batch": 2}
]

ids = executor.insert_many(test_docs)
print("Inserted many IDs:", ids)

print("After inserts:", executor.count_documents())

deleted = executor.delete_one({"test_flag": True})
print("Deleted:", deleted)

print(executor.select({"test_flag": True}))

executor.delete_one({"batch": 1})
executor.delete_one({"batch": 2})

executor.count_documents()

Initial count: 12206
Inserted ID: 6a14e54794309134984a9cc8
[{'_id': ObjectId('6a14e54794309134984a9cc8'), 'test_flag': True, 'value': 123}]
After update: [{'_id': ObjectId('6a14e54794309134984a9cc8'), 'test_flag': True, 'value': 999}]
Inserted many IDs: [ObjectId('6a14e54794309134984a9cc9'), ObjectId('6a14e54794309134984a9cca')]
After inserts: 12209
Deleted: 1
[{'_id': ObjectId('6a14e54794309134984a9cc9'), 'test_flag': True, 'batch': 1}, {'_id': ObjectId('6a14e54794309134984a9cca'), 'test_flag': True, 'batch': 2}]


12206